In [1]:
from cVAE import cVAE, CombineLoss
import pandas as pd
from tortreinador.utils.preprocessing import get_dataloader, load_data, ScalerConfig
from tortreinador.train import TorchTrainer, config_generator
from tortreinador.utils.View import init_weights, split_weights
import torch
from tortreinador.utils.metrics import r2_score
from tortreinador.utils.Recorder import MetricManager, MetricDefine
import os
from config import input_parameters, output_parameters, BASE_PATH
import numpy as np

In [2]:
df_chunk_0 = pd.read_parquet(".\\data\\data_chunk_0.parquet")
df_chunk_1 = pd.read_parquet(".\\data\\data_chunk_1.parquet")

df_all = pd.concat([df_chunk_0, df_chunk_1])

In [4]:
t_loader, v_loader, test_x, test_y, s_x, s_y = load_data(data=df_all, input_parameters=input_parameters,
                                                         output_parameters=output_parameters,
                                                         normal=ScalerConfig(on=True, method='standard', normal_y=True), if_shuffle=True, batch_size=1024, num_workers=8, add_noise=True, error_rate=[0.14, 0.04, 0.12, 0.13], only_noise=True)

In [ ]:
output_dim = len(output_parameters)
input_dim = len(input_parameters)
model_hps = {
        'c_dim': input_dim,
        'z_dim': 72,
        'o_dim': output_dim,
        'num_hidden': 512,
        'mode': 'condition',
        # 'ppa_hidden': 128,
    }

optim_hps = {
    'lr': 0.0001984,
    'weight_decay': 0.00001
}

dataset_hps = {
    'batch_size': 1024,
    'shuffle': True,
    'num_workers': 0,
    'pin_memory':torch.cuda.is_available()
}

trainer_hps = {
    'epoch': 10,
    'metric_manager': MetricManager([MetricDefine(metric_name='train_loss', metric_mode=1, use_as_criterion=True),
                                     MetricDefine(metric_name='train_mse', metric_mode=1),
                                     # MetricDefine(metric_name='beta', metric_mode=1),
                                     MetricDefine(metric_name='train_kld', metric_mode=1),
                                     MetricDefine(metric_name='train_r2', metric_mode=1),
                                     # MetricDefine(metric_name='val_mmd', metric_mode=2),
                                     MetricDefine(metric_name='val_mse', metric_mode=2, use_as_valloss=True),
                                     MetricDefine(metric_name='val_r2', metric_mode=2, use_as_baseline=True)]),
    'data_save_mode': 'csv'
}
# MetricDefine(metric_name='CD', metric_mode=2)

trainer_configs = {
    'warmup_epochs': 10,
    'best_loss': 0.6,
    'auto_save': 5,
    'validation_cycle': 1,
    'logger_on': True,
    # 'model_save_path': 'E:\\Resources\\scun',
    # 'logger_dir': 'E:\\Resources\\scun',
    'model_save_path': os.path.join(BASE_PATH, "cVAE"),
    'logger_dir': os.path.join(BASE_PATH, "cVAE"),
}

In [5]:
# Model
model = cVAE(i_dim=len(output_parameters), z_dim=int(len(output_parameters) * 7), num_hidden=1024,
                  c_dim=len(input_parameters), o_dim=len(output_parameters))
init_weights(model)
criterion = CombineLoss()
optimizer = torch.optim.Adam(split_weights(model), **optim_hps)

In [6]:
class Trainer(TorchTrainer):

    def calculate(self, x, y, mode=1):
        '''Single prediction'''
        B = x.shape[0]


        if mode == 1:
            recon_x, l_z_mu, l_z_logvar = self.model(y, x)

            loss, re_loss, kld = self.criterion(recon_x, y, l_z_mu, l_z_logvar, auto_adj=(0.02, 0.03))    # z, mu, logvar min(0.5 + self.current_epoch * 0.1, 1.5)
            metric_per = r2_score(y, recon_x)

            return_dict = {
                    'train_loss': loss,
                    'train_mse': re_loss,
                    # 'beta': beta,
                    'train_kld': kld,
                    'train_r2': metric_per,
                }

        else:
            fake_z = torch.from_numpy(np.random.normal(0, 1, (B, model.z_dim))).to(torch.float).to(self.device)
            fake_y = torch.from_numpy(np.random.normal(0, 1, (B, 8))).to(torch.float).to(self.device)
            l_z_mu, l_z_logvar, z = self.model.encoder(fake_y, x)
            recon_x = self.model.decoder(fake_z, x)
            loss, re_loss, kld = self.criterion(recon_x, y, l_z_mu, l_z_logvar, auto_adj=(0.02, 0.03))
            metric_per = r2_score(y, recon_x)

            return_dict = {
                    'val_mse': re_loss,
                    'val_r2': metric_per,
                }

        return self._standard_return(mode=mode, update_values=return_dict)

In [7]:
trainer = Trainer(is_gpu=True, model=model, criterion=criterion, optimizer=optimizer, **trainer_hps)
config = config_generator(**trainer_configs)

Epoch:200, Device: cuda


In [8]:
result = trainer.fit(t_loader, v_loader, **config)